# Feature Engineering

<div style="text-align: justify">

The following notebook is dedicated to feature engineering for the  <b> Tau Supersymmetry </b> search analysis. Feature engineering involves preparing input features to enhance their quality and relevance for the ML-based analysis. A general data processing workflow is structured as follows:

</div>

<img src="../assets/data_processing.png" alt="Workflow" style="width: 60%; display: block; margin: 0 auto;"/>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis configuration |
| Load | `io.load_samples` | Read per-sample parquet files |
| Group | `merger.group_samples` | Categorise into data / background / signal |
| Label | `features.assign_event_origin` | Tag each event with its sample name |
| Merge | `merger.merge_signals` | Consolidate signal samples by strategy |
| Split | `merger.split_mc_data` | Separate MC (background + signal) from data |
| Class | `merger.assign_class` | Assign integer class labels |
| Flatten | `merger.dict_to_array` | Concatenate all samples into one array |
| Extract | `features.extract_feature_from_array` | Save eventOrigin and tau_n before deletion |
| Drop | `features.drop_features` | Remove cleaning, truth, weight and auxiliary fields |
| Rectify | `rectangularizer.rectangularize_pad_array` | Pad jagged features and convert to DataFrame |
| Align | — | Restrict data columns to match MC |
| Fill | `rectangularizer.fill_padding` | Replace NaN padding values |
| Reattach | — | Add eventOrigin and tau_n back to MC DataFrame |
| Weights | `features.assign_class_weights` | Compute and attach per-class sample weights to MC |
| Save | `io.save_dataframe` | Write mc.parquet and data.parquet |

The same pipeline is available as a CLI via `python feature_engineer.py` or `make feature-engineer`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Awkward Array](https://awkward-array.readthedocs.io/en/latest/)
* [Pandas](https://pandas.pydata.org/)

Serialization:
* [Apache Parquet](https://parquet.apache.org/)

### Notebook

Activating autoreload of imported modules.

In [1]:
%load_ext autoreload
%autoreload 2

Initializing the project root.

In [2]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [3]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

Unessential warnings suppressed.
ATLAS style applied with LaTeX.


## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, samples, features) are defined in `configs/` and can be overridden here.

In [4]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config")

Resolving the input directory from config.

In [5]:
from src.processing.analysis import get_output_paths

input_dir = path / get_output_paths(cfg)["samples_dir"]

## Deserialization

Loading all per-sample parquet files produced by the preprocessing pipeline.

In [6]:
from src.processing.io import load_samples

sample_ids = [f.stem for f in input_dir.glob("*.parquet")]
samples = load_samples(input_dir, sample_ids)

Grouping samples into data, background, and signal categories.

In [10]:
from src.processing.merger import group_samples

grouped = group_samples(samples, cfg)

## Feature Engineering

Tagging each event with its sample name for later use in analysis and plotting.

In [12]:
from src.processing.features import assign_event_origin

assign_event_origin(grouped)         

### Signal Merger

Merging signal samples.

In [14]:
from src.processing.merger import merge_signals, split_mc_data

grouped["signal"] = merge_signals(grouped["signal"], cfg)

Splitting into Monte-Carlo (background + signal) and data. Signal strategy is controlled by `configs/merge/default.yaml` (`as_one`, `as_type`, or `as_mass`).

In [16]:
samples_mc, samples_data = split_mc_data(grouped)

### Class

Assigning an integer class label to each MC sample for ML training.

In [19]:
from src.processing.merger import assign_class                                        

assign_class(samples_mc)   

Concatenating all samples into a single flat array.

In [ ]:
from src.processing.merger import dict_to_array

samples_mc = dict_to_array(samples_mc)
samples_data = dict_to_array(samples_data)

### Feature Extraction

Extracting features that will be needed after rectangularization.

In [24]:
from src.processing.features import extract_feature_from_array

event_origin = extract_feature_from_array(
    array_in=samples_mc,
    feature_name="eventOrigin",
)

tau_n = extract_feature_from_array(
    array_in=samples_mc,
    feature_name="tau_n",
)

### Feature Deletion

Deleting cleaning, weight, truth, and auxiliary features.

In [ ]:
from src.processing.features import drop_features, resolve_features_to_drop

features_to_drop = resolve_features_to_drop(cfg)

samples_mc = drop_features(array_in=samples_mc, feature_list=features_to_drop)
samples_data = drop_features(array_in=samples_data, feature_list=features_to_drop)

### Rectangularization & Padding

Kinematic data (mainly momenta and related quantities) is heavily nested / jagged. Most of the ML algorithms do not work well with such complicated arrays. Hence, the rectangular format is preferred. The current solution is to set up a certain threshold for a number of entries (jets) each event can take and pad them with zeros. An example is presented below:

<img src="../assets/padding_scheme.png" alt="Workflow" style="width: 60%; display: block; margin: 0 auto;"/>

Rectangularizing with parameters from `configs/data/default.yaml`:
* padding depth: `cfg.data.padding_threshold` (default 3 particles),
* MC: keeping features with at least `cfg.data.nan_threshold` (default 50%) non-missing values,
* data: keeping all features, then aligning to the MC column set.

In [26]:
from src.processing.rectangularizer import rectangularize_pad_array

In [ ]:
df_mc = rectangularize_pad_array(
    array_in=samples_mc,
    padding_threshold=cfg.data.padding_threshold,
    nan_threshold=cfg.data.nan_threshold,
)

In [ ]:
df_data = rectangularize_pad_array(
    array_in=samples_data,
    padding_threshold=cfg.data.padding_threshold,
    nan_threshold=0.0,
)

Aligning data columns to MC.

In [ ]:
df_data = df_data[df_mc.columns.intersection(df_data.columns)]

### Padding Fill

Replacing NaN padding values with a fixed value or per-column mean.

In [ ]:
from src.processing.rectangularizer import fill_padding

df_mc = fill_padding(df=df_mc, strategy="0")
df_data = fill_padding(df=df_data, strategy="0")

### Reattach Metadata

Attaching `eventOrigin` and `tau_n` back to the MC DataFrame — excluded from ML training but needed for downstream analysis and plotting.

In [ ]:
df_mc['eventOrigin'] = event_origin
df_mc['tau_n'] = tau_n

### Class Weights

Computing per-class sample weights from the full MC dataset to correct for class imbalance during training.

In [ ]:
from src.processing.features import assign_class_weights

assign_class_weights(df_mc)

### Preview

Previewing the processed rectangular data as a dataframe.

In [ ]:
df_mc

In [ ]:
df_data

## Serialization

Saving MC and data dataframes as parquet files to the dataframes directory.

In [ ]:
from src.processing.analysis import get_output_paths
from src.processing.io import save_dataframe

dataframes_dir = path / get_output_paths(cfg)["dataframes_dir"]

In [ ]:
save_dataframe(df=df_mc, path=dataframes_dir / "mc.parquet")
save_dataframe(df=df_data, path=dataframes_dir / "data.parquet")